In [1]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import tensorflow as tf
import joblib
import random

FEATURE_ORDER = ['age_at_visit', 'male', 'bmi', 'hdl', 'chol', 'sbp', 'dbp', 'smoke', 'fib4']

scaler = joblib.load('scaler.pkl')

X_test = np.load("X_test_variable.npy", allow_pickle=True)
y_test = np.load("y_test_variable.npy", allow_pickle=True)

nafld_positive, nafld_negetive=list(), list()
for index, value in enumerate(y_test):
    if value == 1:
        nafld_positive.append(index)
    else:
        nafld_negetive.append(index)

In [20]:
index_found = random.choice(nafld_negetive)
scaled_sample = X_test[index_found]
real_data_scaled = scaled_sample[np.any(scaled_sample != 0, axis=1)]
original_data = scaler.inverse_transform(real_data_scaled)
np.set_printoptions(precision=2, suppress=True)

print("Features:", FEATURE_ORDER)
print(original_data)

model = load_model("phase3_var_lstm_1.keras")
sample_to_test=X_test[index_found]
sample_for_prediction = np.expand_dims(sample_to_test, axis=0)
prediction_raw = model.predict(sample_for_prediction)[0][0]
print(prediction_raw)

Features: ['age_at_visit', 'male', 'bmi', 'hdl', 'chol', 'sbp', 'dbp', 'smoke', 'fib4']
[[ 70.99   1.    30.97  37.    37.   132.   132.     1.     3.12]
 [ 71.58   1.    30.97  38.    38.   132.   132.     1.     3.12]
 [ 71.95   1.    30.97  38.    38.   148.   148.     1.     3.12]
 [ 72.1    1.    30.97  36.    36.   148.   148.     1.     3.12]
 [ 72.39   1.    30.97  40.    40.   148.   148.     1.     3.12]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/step
0.00035284492


In [60]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import joblib
import pandas as pd

MAX_TIMESTEPS = 6  
N_FEATURES = 9    
FEATURE_ORDER = ['age_at_visit', 'male', 'bmi', 'hdl', 'chol', 'sbp', 'dbp', 'smoke', 'fib4']

model = load_model("phase3_var_lstm_1.keras")
scaler = joblib.load('scaler.pkl')

my_patient_data = [
    [ 65.28, 0.  , 44.69, 59.  , 59.  , 146.  , 146.  , 0.  , 21.56],
    [ 66.28, 0.  , 44.69, 55.  , 55.  , 146.  , 146.  , 0.  , 21.56],
    [ 66.56, 0.  , 44.69, 55.  , 55.  , 146.  , 146.  , 1.  , 21.56],
    [ 66.56, 0.  , 44.69, 55.  , 55.  , 146.  , 146.  , 0.  , 21.56],
    [ 66.81, 0.  , 44.69, 55.  , 55.  , 146.  , 146.  , 0.  , 21.56]
]

my_patient_data_df = pd.DataFrame(my_patient_data, columns=FEATURE_ORDER)
my_patient_data_scaled = scaler.transform(my_patient_data_df)


# It pads the data to shape (1, 6, 9)
my_patient_data_padded = pad_sequences(
    [my_patient_data_scaled], 
    maxlen=MAX_TIMESTEPS, 
    padding='pre', 
    dtype='float32', 
    value=0.0
)

prediction_raw = model.predict(my_patient_data_padded)[0][0]
prediction_percentage = prediction_raw * 100

print("\n--- FINAL PREDICTION ---")
print(prediction_percentage)

if prediction_percentage > 50:
    print("Prediction: Positive for NAFLD at their next visit. ")
else:
    print("Prediction: Negative for NAFLD at their next visit.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 734ms/step

--- FINAL PREDICTION ---
99.66771
Prediction: Positive for NAFLD at their next visit. 
